# listens to Kafka and updates is_active in t1

In [6]:
import psycopg2
import uuid
from confluent_kafka import Consumer

In [7]:
#This cell is used to connect to the postgres database

connection=psycopg2.connect(
    host="localhost",
    database="pdf_pipeline",
    user="postgres",
    port="5432",
    password="root"
)
cursor=connection.cursor()

In [8]:
#This cell is used to define the consumer and connect to the kafka server

consumer=Consumer({'bootstrap.servers':"localhost:9092",
                   'group.id':"pdf_consumer",
                   'auto.offset.reset':'earliest'})

In [9]:
consumer.subscribe(["pdf_processed"])

In [ ]:
while True:
    message=consumer.poll(1.0)
    if message is None:
        continue
    if message.error():
        print("Error: {}".format(message.error()))
    else:
        uuid=message.value().decode("utf-8")    #since kafka stores messages as bytes, we need to decode it to a string
        cursor.execute("UPDATE t1 SET is_active=False WHERE id=%s",(uuid,))
        connection.commit()